# Road Accident Risk Prediction
Predicts whether a road segment is **high risk** or **low risk** for an accident, using `train.csv`. Each step below is its own cell — run them in order (Shift+Enter).


## 1. Install & import libraries

In [ ]:
!pip install -q scikit-learn xgboost joblib


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import joblib

RANDOM_STATE = 42


## 2. Upload and load `train.csv`
Run this cell, then click **Choose Files** and select `train.csv` from your computer.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select train.csv here


In [ ]:
df = pd.read_csv('train.csv')
print(df.shape)
df.head()


## 3. Explore the data

In [ ]:
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())
print("\naccident_risk summary:")
print(df['accident_risk'].describe())


In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(df['accident_risk'], bins=40, kde=True)
plt.axvline(0.5, color='red', linestyle='--', label='threshold = 0.5')
plt.title('Distribution of accident_risk')
plt.legend()
plt.show()


## 4. Turn `accident_risk` into a classification label
`accident_risk` is a continuous score (0-1) in the raw data. To get accuracy/F1 (classification metrics), we label anything **>= 0.5 as High Risk (1)** and the rest as **Low Risk (0)**.
Change `THRESHOLD` below if you want a stricter/looser cutoff.

In [ ]:
THRESHOLD = 0.5
df['risk_label'] = (df['accident_risk'] >= THRESHOLD).astype(int)
print(df['risk_label'].value_counts(normalize=True))


## 5. Encode categorical features

In [ ]:
categorical_cols = ['road_type', 'lighting', 'weather', 'time_of_day']
boolean_cols = ['road_signs_present', 'public_road', 'holiday', 'school_season']
numeric_cols = ['num_lanes', 'curvature', 'speed_limit', 'num_reported_accidents']

df_model = df.copy()
for col in boolean_cols:
    df_model[col] = df_model[col].astype(int)

df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=False)

feature_cols = numeric_cols + boolean_cols + [c for c in df_model.columns
                if any(c.startswith(cat + '_') for cat in categorical_cols)]

X = df_model[feature_cols]
y = df_model['risk_label']
print(X.shape, y.shape)
X.head()


## 6. Train / validation split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, " Validation:", X_val.shape)


## 7. Train a baseline model (Logistic Regression)

In [ ]:
baseline_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
baseline_model.fit(X_train, y_train)
baseline_preds = baseline_model.predict(X_val)
print("Baseline trained.")


## 8. Evaluate baseline: accuracy, F1, confusion matrix

In [ ]:
acc = accuracy_score(y_val, baseline_preds)
f1 = f1_score(y_val, baseline_preds)
precision = precision_score(y_val, baseline_preds)
recall = recall_score(y_val, baseline_preds)

print(f"Accuracy:  {acc:.4f}")
print(f"F1 score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print()
print(classification_report(y_val, baseline_preds, target_names=['Low Risk', 'High Risk']))

cm = confusion_matrix(y_val, baseline_preds)
ConfusionMatrixDisplay(cm, display_labels=['Low Risk', 'High Risk']).plot(cmap='Blues')
plt.title('Baseline (Logistic Regression) - Confusion Matrix')
plt.show()


## 9. Train a stronger model (Random Forest)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=None, random_state=RANDOM_STATE, n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_val)

print(f"Accuracy:  {accuracy_score(y_val, rf_preds):.4f}")
print(f"F1 score:  {f1_score(y_val, rf_preds):.4f}")
print(classification_report(y_val, rf_preds, target_names=['Low Risk', 'High Risk']))


## 10. Train XGBoost (usually the best performer)

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.08,
    subsample=0.9, colsample_bytree=0.9,
    random_state=RANDOM_STATE, eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_val)

print(f"Accuracy:  {accuracy_score(y_val, xgb_preds):.4f}")
print(f"F1 score:  {f1_score(y_val, xgb_preds):.4f}")
print(classification_report(y_val, xgb_preds, target_names=['Low Risk', 'High Risk']))

cm = confusion_matrix(y_val, xgb_preds)
ConfusionMatrixDisplay(cm, display_labels=['Low Risk', 'High Risk']).plot(cmap='Greens')
plt.title('XGBoost - Confusion Matrix')
plt.show()


## 11. Compare all models side by side

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y_val, baseline_preds),
        accuracy_score(y_val, rf_preds),
        accuracy_score(y_val, xgb_preds),
    ],
    'F1 Score': [
        f1_score(y_val, baseline_preds),
        f1_score(y_val, rf_preds),
        f1_score(y_val, xgb_preds),
    ],
})
results = results.sort_values('F1 Score', ascending=False).reset_index(drop=True)
results


In [ ]:
best_model_name = results.iloc[0]['Model']
best_model = {'Logistic Regression': baseline_model,
              'Random Forest': rf_model,
              'XGBoost': xgb_model}[best_model_name]
print(f"Best model: {best_model_name}")


## 12. Feature importance (which factors matter most)

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importance = pd.Series(best_model.feature_importances_, index=feature_cols)
    importance = importance.sort_values(ascending=False).head(15)
    plt.figure(figsize=(8,5))
    importance.plot(kind='barh')
    plt.gca().invert_yaxis()
    plt.title(f'Top Feature Importances ({best_model_name})')
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance not available for this model type.")


## 13. (Optional) Predict on `test.csv`
If you have a `test.csv` (same feature columns, no `accident_risk`), upload and predict here.

In [ ]:
from google.colab import files
test_uploaded = files.upload()  # select test.csv here (skip this cell if you don't have one)


In [ ]:
test_df = pd.read_csv('test.csv')
test_ids = test_df['id']

test_model_df = test_df.copy()
for col in boolean_cols:
    test_model_df[col] = test_model_df[col].astype(int)
test_model_df = pd.get_dummies(test_model_df, columns=categorical_cols, drop_first=False)

# make sure test has exactly the same columns as training data
for col in feature_cols:
    if col not in test_model_df.columns:
        test_model_df[col] = 0
X_test = test_model_df[feature_cols]

test_preds = best_model.predict(X_test)
test_probs = best_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'id': test_ids,
    'risk_label': test_preds,
    'accident_risk_probability': test_probs
})
submission.to_csv('my_submission.csv', index=False)
files.download('my_submission.csv')
submission.head()


## 14. Save the trained model for the VS Code app

In [ ]:
model_bundle = {
    'model': best_model,
    'feature_cols': feature_cols,
    'categorical_cols': categorical_cols,
    'boolean_cols': boolean_cols,
    'numeric_cols': numeric_cols,
    'threshold': THRESHOLD,
    'model_name': best_model_name,
    'val_accuracy': float(results.iloc[0]['Accuracy']),
    'val_f1': float(results.iloc[0]['F1 Score']),
}
joblib.dump(model_bundle, 'accident_risk_model.pkl')
print("Saved accident_risk_model.pkl")


In [ ]:
from google.colab import files
files.download('accident_risk_model.pkl')


### Next step
Download `accident_risk_model.pkl` (the cell above triggers the download), then move it into your VS Code project folder next to `app.py` to run the interactive interface locally.